# TA1: Predicting Stock Prices Using RNN with LSTM

Name : Samiksha Chavan

Class : TY AIDS A08



**Objective:** Develop a Recurrent Neural Network (RNN) with Long Short-Term Memory (LSTM) units in Python to predict future stock prices based on historical data.

**Stock Selected:** Apple Inc. (AAPL) — S&P 500 constituent  
**Period:** 2019 to 2024 (5 years of historical data)

## Task 1: Data Preparation
### 1.1 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

# Data download
import yfinance as yf

# Scikit-learn for preprocessing and metrics
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# TensorFlow / Keras for LSTM model
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")
print(f"Pandas version     : {pd.__version__}")

### 1.2 Download Historical Stock Data

In [ ]:
TICKER     = 'AAPL'
START_DATE = '2019-01-01'
END_DATE   = '2024-12-31'

print(f"Downloading {TICKER} data from {START_DATE} to {END_DATE} ...")
df_raw = yf.download(TICKER, start=START_DATE, end=END_DATE, auto_adjust=True)
df_raw.reset_index(inplace=True)

print(f"\nShape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head(10)

### 1.3 Exploratory Data Analysis (EDA)

In [ ]:
print("=== Basic Info ===")
print(df_raw.info())
print("\n=== Descriptive Statistics ===")
print(df_raw.describe())
print("\n=== Missing Values ===")
print(df_raw.isnull().sum())

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Close price
axes[0].plot(df_raw['Date'], df_raw['Close'], color='steelblue', linewidth=1.2)
axes[0].set_title(f'{TICKER} Closing Price (2019–2024)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Price (USD)')
axes[0].grid(alpha=0.3)

# Volume
axes[1].bar(df_raw['Date'], df_raw['Volume'], color='coral', alpha=0.6, width=1)
axes[1].set_title(f'{TICKER} Daily Trading Volume', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Volume')
axes[1].grid(alpha=0.3)

# High-Low spread
axes[2].fill_between(df_raw['Date'], df_raw['Low'], df_raw['High'],
                     alpha=0.4, color='green', label='High-Low Range')
axes[2].plot(df_raw['Date'], df_raw['Close'], color='darkgreen',
             linewidth=1, label='Close')
axes[2].set_title(f'{TICKER} Daily High-Low Range vs Close', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Price (USD)')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 1.4 Feature Engineering & Selection

In [ ]:
df = df_raw.copy()

# ---- Technical indicators ----
# 1) 10-day Simple Moving Average
df['SMA_10'] = df['Close'].rolling(window=10).mean()

# 2) 30-day Simple Moving Average
df['SMA_30'] = df['Close'].rolling(window=30).mean()

# 3) Daily Return (%)
df['Daily_Return'] = df['Close'].pct_change() * 100

# 4) Price Range (intra-day volatility)
df['Price_Range'] = df['High'] - df['Low']

# Drop NaN rows introduced by rolling windows
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

# Selected features for the model
FEATURES = ['Open', 'High', 'Low', 'Close', 'Volume',
            'SMA_10', 'SMA_30', 'Daily_Return', 'Price_Range']

print(f"Features selected : {FEATURES}")
print(f"Dataset shape after feature engineering : {df[FEATURES].shape}")
df[FEATURES].head()

### 1.5 Normalization & Scaling

In [ ]:
# We scale each feature independently to [0, 1]
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(df[FEATURES])

# Separate scaler for the target (Close price) for inverse-transform later
close_idx = FEATURES.index('Close')
scaler_close = MinMaxScaler(feature_range=(0, 1))
scaler_close.fit(df[['Close']])

print(f"Scaled data shape : {data_scaled.shape}")
print(f"Min values : {data_scaled.min(axis=0).round(4)}")
print(f"Max values : {data_scaled.max(axis=0).round(4)}")

### 1.6 Create Sequences for LSTM (Train / Test Split)

In [ ]:
SEQUENCE_LENGTH = 60   # look-back window (60 trading days ≈ 3 months)
TRAIN_RATIO     = 0.80

def create_sequences(data, seq_len, target_col_idx):
    """Convert scaled 2-D array into (X, y) sequences."""
    X, y = [], []
    for i in range(seq_len, len(data)):
        X.append(data[i - seq_len:i, :])          # all features
        y.append(data[i, target_col_idx])          # next Close price
    return np.array(X), np.array(y)

X_all, y_all = create_sequences(data_scaled, SEQUENCE_LENGTH, close_idx)

split = int(len(X_all) * TRAIN_RATIO)
X_train, X_test = X_all[:split], X_all[split:]
y_train, y_test = y_all[:split], y_all[split:]

print(f"Total sequences   : {len(X_all)}")
print(f"Training samples  : {X_train.shape}  →  y_train: {y_train.shape}")
print(f"Testing  samples  : {X_test.shape}   →  y_test : {y_test.shape}")

---

## Task 2: Model Development — RNN with LSTM

In [ ]:
n_features = X_train.shape[2]   # number of input features

def build_lstm_model(seq_len, n_feat,
                     units_1=128, units_2=64, units_3=32,
                     dropout=0.2, learning_rate=1e-3):
    """
    Stacked LSTM model:
      - Layer 1 : LSTM (return_sequences=True)  + Dropout
      - Layer 2 : LSTM (return_sequences=True)  + Dropout
      - Layer 3 : LSTM (return_sequences=False) + Dropout
      - Dense   : 1 output neuron (next closing price)
    """
    model = Sequential([
        # ---- LSTM Block 1 ----
        LSTM(units_1, return_sequences=True,
             input_shape=(seq_len, n_feat)),
        Dropout(dropout),

        # ---- LSTM Block 2 ----
        LSTM(units_2, return_sequences=True),
        Dropout(dropout),

        # ---- LSTM Block 3 ----
        LSTM(units_3, return_sequences=False),
        Dropout(dropout),

        # ---- Output layer ----
        Dense(1)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mean_squared_error'
    )
    return model

model = build_lstm_model(SEQUENCE_LENGTH, n_features)
model.summary()

---

## Task 3: Training

In [ ]:
EPOCHS     = 100
BATCH_SIZE = 32

# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss', patience=10,
    restore_best_weights=True, verbose=1
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5,
    patience=5, min_lr=1e-6, verbose=1
)

history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print(f"\nTraining stopped at epoch : {len(history.history['loss'])}")
print(f"Best val_loss             : {min(history.history['val_loss']):.6f}")

In [ ]:
# Plot Training & Validation Loss
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'],     label='Training Loss',   color='steelblue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='coral')
plt.title('Model Loss During Training', fontsize=13, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## Task 4: Prediction & Evaluation

In [ ]:
# ---- Predict on test set ----
y_pred_scaled = model.predict(X_test, verbose=0)

# Inverse-transform back to USD
y_pred = scaler_close.inverse_transform(y_pred_scaled)
y_true = scaler_close.inverse_transform(y_test.reshape(-1, 1))

print(f"y_pred shape : {y_pred.shape}")
print(f"y_true shape : {y_true.shape}")

### 4.1 Performance Metrics

In [ ]:
mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print("=" * 40)
print("       MODEL PERFORMANCE METRICS")
print("=" * 40)
print(f"  MAE  (Mean Absolute Error)          : ${mae:.2f}")
print(f"  RMSE (Root Mean Squared Error)      : ${rmse:.2f}")
print(f"  MAPE (Mean Abs Percentage Error)    : {mape:.2f}%")
print("=" * 40)

### 4.2 Actual vs Predicted — Line Chart

In [ ]:
# Dates for the test window
test_dates = df['Date'].values[SEQUENCE_LENGTH + split:]

plt.figure(figsize=(14, 5))
plt.plot(test_dates, y_true,  label='Actual Price',    color='steelblue', linewidth=1.5)
plt.plot(test_dates, y_pred,  label='Predicted Price', color='tomato',    linewidth=1.5, linestyle='--')
plt.title(f'{TICKER} — Actual vs LSTM Predicted Closing Price (Test Set)',
          fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4.3 Full History Context Plot

In [ ]:
train_dates = df['Date'].values[SEQUENCE_LENGTH:SEQUENCE_LENGTH + split]
y_train_pred_scaled = model.predict(X_train, verbose=0)
y_train_pred = scaler_close.inverse_transform(y_train_pred_scaled)
y_train_true = scaler_close.inverse_transform(y_train.reshape(-1, 1))

plt.figure(figsize=(16, 5))
plt.plot(train_dates, y_train_true, color='royalblue',  linewidth=1,   label='Train Actual')
plt.plot(train_dates, y_train_pred, color='dodgerblue', linewidth=1,   linestyle='--', label='Train Predicted')
plt.plot(test_dates,  y_true,       color='green',      linewidth=1.2, label='Test Actual')
plt.plot(test_dates,  y_pred,       color='red',        linewidth=1.2, linestyle='--', label='Test Predicted')
plt.axvline(x=test_dates[0], color='black', linestyle=':', linewidth=1.5, label='Train/Test Split')
plt.title(f'{TICKER} — Full Period: LSTM Actual vs Predicted', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4.4 Forecast Future Stock Prices from a Given Initial Price

In [ ]:
def forecast_future(model, last_sequence, n_days, scaler, scaler_close, close_idx):
    """
    Iteratively predict the next `n_days` closing prices.
    
    Parameters
    ----------
    last_sequence : np.ndarray  shape (seq_len, n_features) — last known scaled window
    n_days        : int         — number of future days to forecast
    """
    predictions = []
    current_seq = last_sequence.copy()   # (seq_len, n_features)

    for _ in range(n_days):
        # Reshape to (1, seq_len, n_features)
        inp = current_seq[np.newaxis, :, :]
        pred_scaled = model.predict(inp, verbose=0)[0, 0]

        # Inverse-transform the Close price prediction
        pred_price = scaler_close.inverse_transform([[pred_scaled]])[0, 0]
        predictions.append(pred_price)

        # Slide window: drop oldest row, append new row
        new_row = current_seq[-1].copy()
        new_row[close_idx] = pred_scaled          # update Close with predicted value
        current_seq = np.vstack([current_seq[1:], new_row])

    return np.array(predictions)


# Use the last 60 days of the test set as the starting window
FORECAST_DAYS = 30
last_seq = X_test[-1]    # shape: (60, n_features)

future_prices = forecast_future(model, last_seq, FORECAST_DAYS,
                                scaler, scaler_close, close_idx)

# Generate future business dates
last_date      = pd.to_datetime(test_dates[-1])
future_dates   = pd.bdate_range(start=last_date + pd.Timedelta(days=1),
                                periods=FORECAST_DAYS)

print(f"Last known date     : {last_date.date()}")
print(f"Last known price    : ${float(y_true[-1]):.2f}")
print(f"Forecast horizon    : {FORECAST_DAYS} trading days")
print(f"\nForecasted Prices:")
for d, p in zip(future_dates, future_prices):
    print(f"  {d.date()}  →  ${p:.2f}")

In [ ]:
# --- Plot Future Forecast ---
CONTEXT_DAYS = 90   # show last 90 actual days for context

plt.figure(figsize=(14, 5))
plt.plot(test_dates[-CONTEXT_DAYS:], y_true[-CONTEXT_DAYS:],
         color='steelblue', linewidth=1.5, label='Actual (Historical)')
plt.plot(test_dates[-CONTEXT_DAYS:], y_pred[-CONTEXT_DAYS:],
         color='cornflowerblue', linewidth=1, linestyle='--', label='LSTM Fit (Test)')
plt.plot(future_dates, future_prices,
         color='tomato', linewidth=2, marker='o', markersize=4, label='Future Forecast')
plt.axvline(x=pd.to_datetime(test_dates[-1]), color='black',
            linestyle=':', linewidth=1.5, label='Forecast Start')
plt.fill_between(future_dates,
                 future_prices * 0.97,   # ±3% uncertainty band
                 future_prices * 1.03,
                 alpha=0.15, color='tomato', label='±3% Uncertainty Band')
plt.title(f'{TICKER} — {FORECAST_DAYS}-Day Future Price Forecast using LSTM',
          fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4.5 Hyperparameter Comparison

In [ ]:
configs = [
    {'units_1': 128, 'units_2': 64,  'units_3': 32, 'dropout': 0.2, 'lr': 1e-3, 'label': 'Default (128-64-32, lr=1e-3)'},
    {'units_1': 64,  'units_2': 32,  'units_3': 16, 'dropout': 0.2, 'lr': 1e-3, 'label': 'Small (64-32-16, lr=1e-3)'},
    {'units_1': 128, 'units_2': 64,  'units_3': 32, 'dropout': 0.3, 'lr': 5e-4, 'label': 'Default + higher dropout + lower lr'},
]

results = []
for cfg in configs:
    m = build_lstm_model(SEQUENCE_LENGTH, n_features,
                         cfg['units_1'], cfg['units_2'], cfg['units_3'],
                         cfg['dropout'], cfg['lr'])
    es = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=0)
    m.fit(X_train, y_train, validation_split=0.1,
          epochs=50, batch_size=32, callbacks=[es], verbose=0)

    yp = scaler_close.inverse_transform(m.predict(X_test, verbose=0))
    mae_  = mean_absolute_error(y_true, yp)
    rmse_ = np.sqrt(mean_squared_error(y_true, yp))
    mape_ = np.mean(np.abs((y_true - yp) / y_true)) * 100
    results.append({'Config': cfg['label'], 'MAE': mae_, 'RMSE': rmse_, 'MAPE (%)': mape_})
    print(f"  [{cfg['label']}]  MAE={mae_:.2f}  RMSE={rmse_:.2f}  MAPE={mape_:.2f}%")

results_df = pd.DataFrame(results)
print("\n")
print(results_df.to_string(index=False))

In [ ]:
# Bar chart: metric comparison across configs
x      = np.arange(len(results_df))
width  = 0.25
labels = [r.split('(')[0].strip() for r in results_df['Config']]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width, results_df['MAE'],      width, label='MAE',     color='steelblue')
ax.bar(x,         results_df['RMSE'],     width, label='RMSE',    color='coral')
ax.bar(x + width, results_df['MAPE (%)'], width, label='MAPE (%)', color='mediumseagreen')

ax.set_title('Hyperparameter Comparison — MAE / RMSE / MAPE', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=10, ha='right')
ax.set_ylabel('Error Value')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---

## Summary & Observations

| Section | Key Points |
|---|---|
| **Data Preparation** | Downloaded 5 years of AAPL data (2019–2024). Added technical indicators: SMA-10, SMA-30, Daily Return, Price Range. MinMaxScaler applied to all 9 features. |
| **Model Architecture** | Stacked 3-layer LSTM (128 → 64 → 32 units) with Dropout (0.2) after each layer to reduce overfitting. Adam optimizer, MSE loss. |
| **Training** | 80/20 train-test split; 10% of training used as validation. EarlyStopping + ReduceLROnPlateau callbacks prevent overfitting and aid convergence. |
| **Performance** | MAE, RMSE, and MAPE reported on the unseen test set. A MAPE < 5% indicates strong predictive accuracy for stock prices. |
| **Future Forecast** | Iterated prediction for 30 future trading days using a sliding-window approach. Uncertainty band (±3%) visualized. |
| **Hyperparameter Study** | Smaller architectures showed slightly higher error; lower learning rate with higher dropout sometimes improved generalization. |

### Limitations & Future Work
- Stock prices are influenced by external factors (news, macroeconomics) not captured in OHLCV data.
- Multi-step iterative forecasting compounds errors — error grows with forecast horizon.
- Future improvements: Attention mechanisms, Transformer models, incorporating sentiment analysis from financial news.